# 31. LoRA Variants Theory | LoRA 变体原理
**难度：** Medium | **环境：** CPU-first | **标签：** `训练微调`, `LoRA`, `低秩适配` | **目标人群：** 训练机制学习者

---

## 本节导读

LoRA 变体看起来很多，但真正要比较的东西并不复杂：它们到底在改什么，代价是什么，换来了什么收益。`10` 先回答 LoRA 本体怎么工作，`26` 再说明量化底座和 LoRA 旁路怎样结合，而 `31` 进一步回答另一个更贴近训练设计的问题：当你已经决定走 adapter 路线后，不同 LoRA 变体该怎么放到同一套比较口径里。

这一节不追求穷举所有 PEFT 名词，而是先把变体选择收敛成三个最小判断：规格字段怎么统一、参数效率和训练稳定性怎么比较、最终推荐为什么必须绑定具体场景。它在训练微调路线里不是松散扩展页，而是项目收口前补链的“方案比较”环节：`26` 先把小显存分支立住，`31` 再回答同样走 adapter 路线时不同 LoRA 变体该怎么比较，后面才接 `32 / 33 / 60`。学完后，你应该能看清“LoRA 本体 -> 量化分支 -> 变体比较 -> 数据与 readiness -> 项目收口”这条主线，而不是停留在名称堆砌。

**关键词：** `rank`, `alpha`, `dropout`, `target modules`

---


## 前置阅读

**导语：** 这一节同时承接 LoRA 本体、训练资源限制和量化微调三条线：先知道 adapter 在改什么，再回来看为什么不同变体会适合不同预算和目标。
- [10. LoRA Tutorial | LoRA 教程](./10_LoRA_Tutorial.ipynb)
- [12. Gradient Accumulation | 梯度累积](./12_Gradient_Accumulation.ipynb)
- [26. QLoRA and 4bit Quantization | QLoRA 与 4-bit 量化](./26_QLoRA_and_4bit_Quantization.ipynb)

## 相关阅读

**导语：** 学完 LoRA 变体原理后，下一步可以沿两条线继续走：一条是项目线，去看这些比较口径如何真正落到微调交付；另一条是选型线，去验证不同变体在 benchmark 和 QLoRA 方案里是否真的值得落地。
- [60. LoRA Fine-Tuning Project | LoRA 微调项目](./60_LoRA_Fine_Tuning_Project.ipynb)
- [63. LoRA Variants Benchmark | LoRA 变体基准对比](./63_LoRA_Variants_Benchmark.ipynb)
- [65. QLoRA Selection Project | QLoRA 方案选择项目](./65_QLoRA_Selection_Project.ipynb)
- [2.3](./2_3.md)

---


### Step 1: 先把变体规格写成统一字段

- 至少记录 `rank`、`alpha`、`dropout` 和 `target_modules`。
- 如果一个变体引入额外门控或动态分配，也要把这些控制开关写出来。
- 统一字段后，后面的对比和项目页才能共享同一套口径。


### Step 2: 比较参数效率、稳定性和复杂度

![LoRA Variant Comparison Map](/02_PyTorch_Algorithms/31_lora_variants_map.svg)

- 参数效率通常来自更小的 rank 或更有选择性的插层。
- 训练稳定性可以通过 loss 波动、是否依赖较强正则来观察。
- 实现复杂度则决定了这个变体适不适合进入真实项目。


### Step 3: 用场景优先级做推荐

- 显存紧张时优先压低训练参数和 target modules 数量。
- 更关心效果时，允许更高 rank 或更多插层，但要接受更高成本。
- 推荐结论应该显式绑定优先级，而不是给出脱离场景的“通用最优”。


### Step 4: 动手实战

1. 补全 `normalize_lora_variant_spec`，把单个变体转成统一规格。
2. 补全 `compare_lora_variants`，输出每个变体的参数效率和综合判断。
3. 补全 `recommend_lora_variant`，根据优先级给出推荐。


### 提示

- `TODO 1` 先把单个变体规格标准化，再补 `trainable_params_ratio` 这类统一字段。
- `TODO 2` 先逐个变体标准化，再给每个变体补一个 `efficiency_score`。
- `TODO 3` 先比较 priority 是 `memory` 还是 `quality`，再按对应口径选推荐项。


In [ ]:
from typing import Dict, List


In [ ]:
def normalize_lora_variant_spec(variant: Dict[str, object]) -> Dict[str, object]:
    """
    TODO 1: 把单个变体转成统一规格。
    """
    # 提示：先规范 target_modules，再计算 trainable_params_ratio，最后返回统一字段字典。
    # target_modules = ???
    # trainable_params_ratio = ???
    raise NotImplementedError


def compare_lora_variants(variants: List[Dict[str, object]]) -> List[Dict[str, object]]:
    """
    TODO 2: 输出每个变体的参数效率和综合判断。
    """
    # 提示：先逐个标准化，再给每个变体补 efficiency_score，最后按高到低排序。
    # compared = ???
    # efficiency_score = ???
    raise NotImplementedError


def recommend_lora_variant(variants: List[Dict[str, object]], priority: str) -> Dict[str, object]:
    """
    TODO 3: 根据优先级给出推荐。
    """
    # 提示：先拿到 compared；若 priority == 'memory'，优先看 trainable_params_ratio；若 priority == 'quality'，优先看 quality_score。
    # compared = ???
    # best = ???
    raise NotImplementedError


In [ ]:
def test_lora_variants_theory_template():
    try:
        variants = [
            {
                'name': 'standard_lora',
                'rank': 8,
                'alpha': 16,
                'dropout': 0.05,
                'target_modules': ['q_proj', 'v_proj'],
                'trainable_params': 4,
                'base_params': 100,
                'quality_score': 0.76,
                'implementation_cost': 0.10,
            },
            {
                'name': 'wide_lora',
                'rank': 16,
                'alpha': 32,
                'dropout': 0.05,
                'target_modules': ['q_proj', 'k_proj', 'v_proj'],
                'trainable_params': 8,
                'base_params': 100,
                'quality_score': 0.82,
                'implementation_cost': 0.18,
            },
        ]
        normalized = normalize_lora_variant_spec(variants[0])
        assert normalized['target_modules'] == ['q_proj', 'v_proj']
        assert abs(normalized['trainable_params_ratio'] - 0.04) < 1e-8
        compared = compare_lora_variants(variants)
        assert len(compared) == 2 and 'efficiency_score' in compared[0]
        assert recommend_lora_variant(variants, priority='memory')['recommended_name'] == 'standard_lora'
        assert recommend_lora_variant(variants, priority='quality')['recommended_name'] == 'wide_lora'
        print('测试通过：LoRA 变体原理模板可以工作。')
    except NotImplementedError:
        raise
    except (AttributeError, NameError, TypeError, ValueError, AssertionError) as e:
        raise NotImplementedError('请先完成 TODO 代码！') from e


test_lora_variants_theory_template()


---

🛑 **STOP HERE** 🛑
<br><br><br><br><br><br><br><br><br><br>
> 请先尝试自己完成代码并跑通测试。<br>
> 如果你正在 Colab 中运行，并且遇到困难没有思路，可以向下滚动查看参考答案。
<br><br><br><br><br><br><br><br><br><br>

---


## 参考代码与解析

### 代码


In [ ]:
def normalize_lora_variant_spec(variant: Dict[str, object]) -> Dict[str, object]:
    """
    TODO 1: 把单个变体转成统一规格。
    """
    # 提示：先规范 target_modules，再计算 trainable_params_ratio，最后返回统一字段字典。
    # target_modules = ???
    # trainable_params_ratio = ???
    target_modules = variant.get('target_modules', [])
    if isinstance(target_modules, str):
        target_modules = [target_modules]
    trainable_params = float(variant.get('trainable_params', 0))
    base_params = float(variant.get('base_params', 0))
    return {
        'name': variant.get('name', 'variant'),
        'rank': int(variant.get('rank', 0)),
        'alpha': int(variant.get('alpha', 0)),
        'dropout': float(variant.get('dropout', 0.0)),
        'target_modules': list(target_modules),
        'trainable_params_ratio': trainable_params / base_params if base_params else 0.0,
        'quality_score': float(variant.get('quality_score', 0.0)),
        'implementation_cost': float(variant.get('implementation_cost', 0.0)),
    }


def compare_lora_variants(variants: List[Dict[str, object]]) -> List[Dict[str, object]]:
    """
    TODO 2: 输出每个变体的参数效率和综合判断。
    """
    # 提示：先逐个标准化，再给每个变体补 efficiency_score，最后按高到低排序。
    # compared = ???
    # efficiency_score = ???
    compared = []
    for variant in variants:
        normalized = normalize_lora_variant_spec(variant)
        compared.append({
            **normalized,
            'efficiency_score': normalized['quality_score'] - normalized['implementation_cost'] - normalized['trainable_params_ratio'],
        })
    return sorted(compared, key=lambda item: item['efficiency_score'], reverse=True)


def recommend_lora_variant(variants: List[Dict[str, object]], priority: str) -> Dict[str, object]:
    """
    TODO 3: 根据优先级给出推荐。
    """
    # 提示：先拿到 compared；若 priority == 'memory'，优先看 trainable_params_ratio；若 priority == 'quality'，优先看 quality_score。
    # compared = ???
    # best = ???
    compared = compare_lora_variants(variants)
    if priority == 'memory':
        best = min(compared, key=lambda item: (item['trainable_params_ratio'], item['implementation_cost']))
    elif priority == 'quality':
        best = max(compared, key=lambda item: (item['quality_score'], -item['implementation_cost']))
    else:
        best = compared[0]
    return {'priority': priority, 'recommended_name': best['name']}


### 解析

**1. TODO 1：把单个变体转成统一规格**
- 先规范 `target_modules` 的格式，再计算 `trainable_params_ratio`，最后把 rank、alpha、dropout 等字段统一收进同一份规格字典。
- 统一规格的意义是先把不同变体放到同一张表上，避免后续比较时口径混乱。

**2. TODO 2：输出每个变体的参数效率和综合判断**
- 先逐个变体标准化，再给每个变体补 `efficiency_score`，最后按高到低排序。
- 这里的 `efficiency_score` 用最小方式把质量、实现成本和训练参数占比合在一起，方便快速比较方案。

**3. TODO 3：根据优先级给出推荐**
- 如果 priority 是 `memory`，优先看 `trainable_params_ratio` 和实现成本；如果是 `quality`，优先看效果分数。
- 推荐结论必须显式绑定场景优先级，否则“通用最优”很难复用到真实项目里。

**4. 这页的定位**
- 先统一规格，后续比较才不会混口径。
- `trainable_params_ratio` 是最常用的参数效率指标。
- 推荐必须显式绑定优先级，否则结论很难复用到别的项目里。
